#### MNIST Veri Seti ile PCA, KNN ve Augmentation Uygulaması

In [27]:
import numpy as np

In [28]:
test_images_path = r"archive\t10k-images.idx3-ubyte"
test_labels_path = r"archive\t10k-labels.idx1-ubyte"

train_images_path = r"archive\train-images.idx3-ubyte"
train_labels_path = r"archive\train-labels.idx1-ubyte"

#### VERİ SETİNİ OKUMA

In [29]:
def read_images(file_path):
    with open(file_path, "rb") as file:
        header = np.frombuffer(file.read(16), dtype=">u4")
        magic_number, image_count, row_count, column_count = header

        pixel_data = file.read()
        pixels = np.frombuffer(pixel_data, dtype=np.uint8)

        images = pixels.reshape(image_count, row_count, column_count)
    
    return images, magic_number, image_count, row_count, column_count

In [30]:
def read_labels(file_path):
    with open(file_path, "rb") as file:
        header = np.frombuffer(file.read(8), dtype=">u4")
        magic_number, label_count = header

        label_data = file.read()
        labels = np.frombuffer(label_data, dtype=np.uint8)

    return labels, magic_number, label_count

In [31]:
train_images, train_image_magic_number, train_image_count ,train_image_row_count, train_image_column_count = read_images(train_images_path)
train_labels, train_label_magic_count, train_label_count = read_labels(train_labels_path)

test_images, test_image_magic_number, test_image_count, test_image_row_count, test_image_column_count = read_images(test_images_path)
test_labels, test_label_magic_count, test_label_count = read_labels(test_labels_path)

print("Train images: ", train_images.shape)
print("Train labels: ", train_labels.shape)
print("test_images: ", test_images.shape)
print("Test labels: ", test_labels.shape)

print("Train image magic number: ", train_image_magic_number)
print("Train label magic number: ", train_label_magic_count)
print("Test image magic number: ", test_image_magic_number)
print("Test label magic number: ", test_label_magic_count)

Train images:  (60000, 28, 28)
Train labels:  (60000,)
test_images:  (10000, 28, 28)
Test labels:  (10000,)
Train image magic number:  2051
Train label magic number:  2049
Test image magic number:  2051
Test label magic number:  2049


#### GÖRÜNTÜLERİ VEKTÖRE ÇEVİRME
- Her bir görsel 28x28'lik bir matristen 784'lük bir vektöre dönüştürülüyor. 

In [32]:
print("Train image: ", train_images.shape)
print("Test image: ", test_images.shape)

train_vectors = train_images.reshape(
    train_images.shape[0], 
    train_images.shape[1] * train_images.shape[2])

test_vectors = test_images.reshape(
    test_images.shape[0],
    test_images.shape[1] * test_images.shape[2]
)

print("Train vectors: ", train_vectors.shape)
print("Test vectors: ", test_vectors.shape)


Train image:  (60000, 28, 28)
Test image:  (10000, 28, 28)
Train vectors:  (60000, 784)
Test vectors:  (10000, 784)


#### NORMALİZASYON

In [33]:
normalized_train_vectors = train_vectors / 255.0
normalized_test_vectors = test_vectors / 255.0

print("Minimum train pixels: ", normalized_train_vectors.min())
print("Maximum train pixel: ", normalized_train_vectors.max())

Minimum train pixels:  0.0
Maximum train pixel:  1.0


#### KNN ile Tek Bir Örnek Tahmini
- Bu fonksiyon, tek bir test görüntüsünün hangi sınıfa ait olduğunu tahmin eder.
- Bunun için test görüntüsü ile eğitim setindeki tüm görüntüler arasındaki uzaklık hesaplanır. 
- Uzaklık hesabında piksel farklarının kareleri toplanır.
    - uzaklık = piksel farklarının kareleri toplamı
- Daha sonra uzaklığı en küçük olan k adet eğitim örneği seçilir.
- Bu örneklerin etiketlerine bakılır ve hangi sınıf en fazla tekrar ediyorsa tahmin olarak o seçilir.

In [34]:
def predict_one(test_vector, train_vectors, train_labels, k):
    distances = np.sum((train_vectors - test_vector) ** 2 , axis=1) 

    nearest_indices = np.argpartition(distances, k-1)[:k]
    nearest_labels = train_labels[nearest_indices]
    label_counts = np.zeros(10, dtype=int)

    for label in nearest_labels:
        label_counts[label] += 1
    
    prediction = np.argmax(label_counts)
    return prediction

#### PCA Hazırlığı

In [35]:
def fit_manual_pca(train_vectors):
    mean_vector = train_vectors.mean(axis=0)
    centered_train_vectors = train_vectors - mean_vector

    sample_count = centered_train_vectors.shape[0]

    # C = (X.T_centered x X_centered) / (n-1)
    covariance_matrix = (centered_train_vectors.T @ centered_train_vectors) / (sample_count - 1)

    eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)
    sorted_indices = np.argsort(eigenvalues)[::-1]

    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[: , sorted_indices]

    return mean_vector, sorted_eigenvalues, sorted_eigenvectors

#### PCA Dönüşümü
- Veriden daha önce hesaplanan ortalama çıkarılır.
- İlk component_count kadar ana bileşen seçilir.
- Veri bu ana bileşenler üzerine projekte edilir.
- Sonuçta daha düşük boyutlu PCA verisi elde edilir.

In [36]:
def transform_manual_pca(vectors, mean_vector, sorted_eigenvectors, component_count):
    centered_vectors = vectors - mean_vector
    principal_components = sorted_eigenvectors[: , : component_count]
    pca_vectors = centered_vectors @ principal_components

    return pca_vectors

#### KNN Değerlendirme Fonksiyonu
- Bütün değerlendirme görüntüleri için `predict_one` fonksiyonu tekrar çağırılır.
- Her görüntü için tahmin yapılır, listeye eklenir ve gerçek etiketler ile karşılaştırılır.
- Kaç tanesinin doğru olduğuna göre `accuracy` hesaplanır.
    - accuracy = doğru tahmin sayısı / toplam örnek sayısı

In [37]:
def evaluate_knn(train_vectors, train_labels, evaluation_vectors, evaluation_labels, k):
    predictions = []

    for vector in evaluation_vectors:
        prediction = predict_one(vector, train_vectors, train_labels, k)
        predictions.append(prediction)
    
    predictions = np.array(predictions)
    accuracy = np.mean(predictions == evaluation_labels)

    return accuracy

#### Train ve Validation Verisi Ayrımı
- Veri setinin karıştırarak train ve validation setlerine ayırmamızın sebebi veri setinin sırasından etkilenmesini engellemektir. 

In [38]:
np.random.seed(42)

indices = np.random.permutation(normalized_train_vectors.shape[0])
validation_count = 10000

validation_indices = indices[:validation_count]
model_train_indices = indices[validation_count:]

model_train_vectors = normalized_train_vectors[model_train_indices]
model_train_labels = train_labels[model_train_indices]

validation_vectors = normalized_train_vectors[validation_indices]
validation_labels = train_labels[validation_indices]

print("Model train vectors shape: ", model_train_vectors.shape)
print("Model train labels shape: ", model_train_labels.shape)

print("Validation vectors shape: ", validation_vectors.shape)
print("Validation labels shape: ", validation_labels.shape)

Model train vectors shape:  (50000, 784)
Model train labels shape:  (50000,)
Validation vectors shape:  (10000, 784)
Validation labels shape:  (10000,)


#### PCA Bilgilerini Train Verisinden Hesaplama

In [39]:
mean_vector, sorted_eigenvalues, sorted_eigenvectors = fit_manual_pca(model_train_vectors)

print("Mean vector shape: ", mean_vector.shape)
print("Sorted eigenvalues shape: ", sorted_eigenvalues.shape)
print("Sorted eigenvectors shape: ", sorted_eigenvectors.shape)

Mean vector shape:  (784,)
Sorted eigenvalues shape:  (784,)
Sorted eigenvectors shape:  (784, 784)


#### Parametre Seçenekleri

In [40]:
component_options = [20, 30, 50, 75, 100, 200]
k_options = [1, 3, 5, 7, 9]

# 10000 adet validation görüntüsünde denemek çok maliyetli olacağı için bunu yapıyoruz
quick_validation_count = 1000

quick_validation_vectors = validation_vectors[: quick_validation_count]
quick_validation_labels = validation_labels[: quick_validation_count]

#### Hızlı Parametre Denemesi (Farklı PCA ve `k` Değerleri Deneniyor)
- Bir üst hücrede denenecek değerler ayarlanmıştı.

In [41]:
results = []

for component_count in component_options:

    model_train_pca_vectors = transform_manual_pca(model_train_vectors , mean_vector, sorted_eigenvectors, component_count)

    quick_validation_pca_vectors = transform_manual_pca(quick_validation_vectors, mean_vector, sorted_eigenvectors, component_count)

    preserved_variance = sorted_eigenvalues[: component_count].sum() / sorted_eigenvalues.sum()

    for k in k_options:

        accuracy = evaluate_knn(model_train_pca_vectors, model_train_labels, quick_validation_pca_vectors, quick_validation_labels, k)

        result = { 
            "component_count" : int(component_count), 
            "k" : int(k), 
            "preserved_variance" : float(preserved_variance * 100), 
            "validation_accuracy" : float(accuracy * 100) 
            }

        results.append(result)

        print(
            "PCA:", component_count,
            "| k:", k,
            "| Preserved variance:", round(preserved_variance * 100, 2),
            "| Accuracy: ", round(accuracy * 100 , 2) 
        )

PCA: 20 | k: 1 | Preserved variance: 64.38 | Accuracy:  96.4
PCA: 20 | k: 3 | Preserved variance: 64.38 | Accuracy:  97.5
PCA: 20 | k: 5 | Preserved variance: 64.38 | Accuracy:  97.0
PCA: 20 | k: 7 | Preserved variance: 64.38 | Accuracy:  96.8
PCA: 20 | k: 9 | Preserved variance: 64.38 | Accuracy:  97.0
PCA: 30 | k: 1 | Preserved variance: 73.08 | Accuracy:  97.5
PCA: 30 | k: 3 | Preserved variance: 73.08 | Accuracy:  97.8
PCA: 30 | k: 5 | Preserved variance: 73.08 | Accuracy:  97.9
PCA: 30 | k: 7 | Preserved variance: 73.08 | Accuracy:  97.8
PCA: 30 | k: 9 | Preserved variance: 73.08 | Accuracy:  97.8
PCA: 50 | k: 1 | Preserved variance: 82.48 | Accuracy:  97.4
PCA: 50 | k: 3 | Preserved variance: 82.48 | Accuracy:  98.2
PCA: 50 | k: 5 | Preserved variance: 82.48 | Accuracy:  97.7
PCA: 50 | k: 7 | Preserved variance: 82.48 | Accuracy:  97.5
PCA: 50 | k: 9 | Preserved variance: 82.48 | Accuracy:  97.3
PCA: 75 | k: 1 | Preserved variance: 88.3 | Accuracy:  97.4
PCA: 75 | k: 3 | Preserve

In [42]:
sorted_results = sorted(results, key=lambda result: result["validation_accuracy"], reverse=True)

for result in sorted_results:
    print(result)

{'component_count': 50, 'k': 3, 'preserved_variance': 82.48486419177762, 'validation_accuracy': 98.2}
{'component_count': 30, 'k': 5, 'preserved_variance': 73.07759581239462, 'validation_accuracy': 97.89999999999999}
{'component_count': 30, 'k': 3, 'preserved_variance': 73.07759581239462, 'validation_accuracy': 97.8}
{'component_count': 30, 'k': 7, 'preserved_variance': 73.07759581239462, 'validation_accuracy': 97.8}
{'component_count': 30, 'k': 9, 'preserved_variance': 73.07759581239462, 'validation_accuracy': 97.8}
{'component_count': 50, 'k': 5, 'preserved_variance': 82.48486419177762, 'validation_accuracy': 97.7}
{'component_count': 75, 'k': 5, 'preserved_variance': 88.30003929785416, 'validation_accuracy': 97.7}
{'component_count': 100, 'k': 1, 'preserved_variance': 91.47383470369031, 'validation_accuracy': 97.7}
{'component_count': 100, 'k': 3, 'preserved_variance': 91.47383470369031, 'validation_accuracy': 97.7}
{'component_count': 75, 'k': 3, 'preserved_variance': 88.3000392978

#### Aday Parametreler
- Her PCA bileşeni sayısı için en iyi 2 sonuç `candidate_parameters` listesine eklenir.
- Bu listede sadece `component_count` ve `k` değerleri tutulur.
- Bu adımın amacı, bütün kombinasyonları tekrar denemek yerine `hızlı validation` sonucuna göre başarılı görünen adaylar ile devam etmektir. 

In [43]:
candidate_parameters = []

for component_count in component_options:

    component_results = []

    for result in results:
        if result["component_count"] == component_count:
            component_results.append(result)
    
    component_results = sorted(component_results, key=lambda result: result["validation_accuracy"], reverse=True)

    for result in component_results[:2]:
        candidate = {"component_count": int(result["component_count"]), "k": int(result["k"])}
        candidate_parameters.append(candidate)

for param in candidate_parameters:
    print(param)

{'component_count': 20, 'k': 3}
{'component_count': 20, 'k': 5}
{'component_count': 30, 'k': 5}
{'component_count': 30, 'k': 3}
{'component_count': 50, 'k': 3}
{'component_count': 50, 'k': 5}
{'component_count': 75, 'k': 5}
{'component_count': 75, 'k': 3}
{'component_count': 100, 'k': 1}
{'component_count': 100, 'k': 3}
{'component_count': 200, 'k': 3}
{'component_count': 200, 'k': 5}


#### Full Validation Denemesi
- Aday parametreler bütün validation verisi ile test edilir.
- Validation accuracy ve korunan varyans hesaplanır.
    - korunan varyans = seçilen özdeğerlerin toplamı / bütün özdeğerlerin toplamı

In [44]:
full_validation_results = []

for parameters in candidate_parameters:
    component_count = parameters["component_count"]
    k = parameters["k"]

    model_train_pca_vectors = transform_manual_pca(model_train_vectors, mean_vector, sorted_eigenvectors, component_count)

    validation_pca_vectors = transform_manual_pca(validation_vectors, mean_vector, sorted_eigenvectors, component_count)

    accuracy = evaluate_knn(model_train_pca_vectors, model_train_labels, validation_pca_vectors, validation_labels, k)

    preserved_variance = sorted_eigenvalues[:component_count].sum() / sorted_eigenvalues.sum()

    result = { 
        "component_count" : int(component_count), 
        "k" : int(k), 
        "preserved_variance" : float(preserved_variance * 100), 
        "validation_accuracy" : float(accuracy * 100) 
    }

    full_validation_results.append(result)

    print(
        "PCA:", component_count,
        "| k:", k,
        "| Preserved variance:", round(preserved_variance * 100, 2),
        "| Accuracy: ", round(accuracy * 100 , 2) 
    )

PCA: 20 | k: 3 | Preserved variance: 64.38 | Accuracy:  96.93
PCA: 20 | k: 5 | Preserved variance: 64.38 | Accuracy:  97.03
PCA: 30 | k: 5 | Preserved variance: 73.08 | Accuracy:  97.56
PCA: 30 | k: 3 | Preserved variance: 73.08 | Accuracy:  97.74
PCA: 50 | k: 3 | Preserved variance: 82.48 | Accuracy:  97.84
PCA: 50 | k: 5 | Preserved variance: 82.48 | Accuracy:  97.65
PCA: 75 | k: 5 | Preserved variance: 88.3 | Accuracy:  97.66
PCA: 75 | k: 3 | Preserved variance: 88.3 | Accuracy:  97.7
PCA: 100 | k: 1 | Preserved variance: 91.47 | Accuracy:  97.6
PCA: 100 | k: 3 | Preserved variance: 91.47 | Accuracy:  97.56
PCA: 200 | k: 3 | Preserved variance: 96.65 | Accuracy:  97.34
PCA: 200 | k: 5 | Preserved variance: 96.65 | Accuracy:  97.34


#### En İyi Parametreler Seçilir
- PCA bileşen sayısı ve KNN için k sayısı belirlenir.
- Bu değerler daha sonra final modelde kullanılacak.

In [45]:
best_result = max(full_validation_results, key=lambda result: result["validation_accuracy"])

best_component_count = best_result["component_count"]
best_k = best_result["k"]

print("Best result: ", best_result)
print("Best component count: ", best_component_count)
print("Best k: ", best_k)

Best result:  {'component_count': 50, 'k': 3, 'preserved_variance': 82.48486419177762, 'validation_accuracy': 97.84}
Best component count:  50
Best k:  3


#### Final PCA Dönüşümü
- PCA bilgiler bütün normalize edilmiş eğitim verisi üzerinde yeniden hesaplanır.
- Bunun sebebi parametre seçimi tamamlandığında eğitim verisinin tamamından yararlanmaktır.
- Sonuçta görüntüler orijinal 784 boyut yerine seçilen en iyi `component_count` kadar özellik ile temsil edilir.

In [46]:
final_mean_vector, final_sorted_eigenvalues, final_sorted_eigenvectors = fit_manual_pca(normalized_train_vectors)

final_train_pca_vectors = transform_manual_pca(normalized_train_vectors, final_mean_vector, final_sorted_eigenvectors, best_component_count)

final_test_pca_vectors = transform_manual_pca(normalized_test_vectors, final_mean_vector, final_sorted_eigenvectors, best_component_count)

#### Baseline Test Sonucu Hesaplanır ve Kaydedilir

In [47]:
baseline_accuracy = evaluate_knn(final_train_pca_vectors, train_labels, final_test_pca_vectors, test_labels, best_k)
print("Final baseline accuracy: ", baseline_accuracy * 100)

final_preserved_variance = final_sorted_eigenvalues[:best_component_count].sum() / final_sorted_eigenvalues.sum()

baseline_result = {
    "dataset" : "original_mnist",
    "component_count": int(best_component_count), 
    "k": int(best_k),
    "preserved_variance" : float(final_preserved_variance * 100),
    "test_accuracy": float(baseline_accuracy * 100)
}
print(baseline_result)

Final baseline accuracy:  97.52
{'dataset': 'original_mnist', 'component_count': 50, 'k': 3, 'preserved_variance': 82.46468633334725, 'test_accuracy': 97.52}


#### Confusion Matrix

In [48]:
def predict_all(train_vectors, train_labels, evaluation_vectors, k):
    predictions = []

    for vector in evaluation_vectors:
        prediction = predict_one(vector, train_vectors, train_labels, k)
        predictions.append(prediction)
    return np.array(predictions)

def calculate_confusion_matrix(true_labels, predicted_labels, class_count=10):
    matrix = np.zeros((class_count, class_count), dtype=int)

    for true_label, predicted_label in zip(true_labels, predicted_labels):
        matrix[int(true_label), int(predicted_label)] += 1
    
    return matrix

baseline_predictions = predict_all(final_train_pca_vectors, train_labels, final_test_pca_vectors, best_k)
baseline_confusion_matrix = calculate_confusion_matrix(test_labels, baseline_predictions)

print("Baseline confusion matrix: ")
print(baseline_confusion_matrix)



Baseline confusion matrix: 
[[ 975    1    1    0    0    1    1    1    0    0]
 [   0 1130    3    0    0    0    1    0    0    1]
 [   5    2 1008    1    1    0    1   12    2    0]
 [   0    0    4  977    1   11    0    7    8    2]
 [   2    1    0    0  959    0    3    1    0   16]
 [   3    0    0   10    2  863    7    1    3    3]
 [   6    3    0    0    2    2  944    0    1    0]
 [   0   14    7    1    3    0    0  996    0    7]
 [   5    0    4   16    5    6    3    3  929    3]
 [   7    3    2    6    9    2    1    3    5  971]]


#### Parçalı Augmented Görüntü Üretme

In [49]:
def create_augmented_image(images, labels, target_label):
    label_indices = np.where(labels == target_label)[0]

    selected_indices = np.random.choice(label_indices, size=4, replace=False)

    image1 = images[selected_indices[0]] 
    image2 = images[selected_indices[1]]
    image3 = images[selected_indices[2]]
    image4 = images[selected_indices[3]]

    top_left = image1[:14 , :14]
    top_right = image2[:14 , 14:]
    bottom_left = image3[14:, :14]
    bottom_right = image4[14: , 14:]

    top_half = np.hstack((top_left, top_right))
    bottom_half = np.hstack((bottom_left, bottom_right))
    augmented_image = np.vstack((top_half, bottom_half))

    augmented_label = target_label
    return augmented_image , augmented_label


#### Augmented Veri Seti Üretme

In [50]:
def generate_augmented_dataset(images, labels, new_images_per_class):
    augmented_images = []
    augmented_labels = []

    for target_label in range(10): # mnist'te 10 tane sınıf var
        for _ in range(new_images_per_class):
            augmented_image, augmented_label = create_augmented_image(images, labels, target_label)

            augmented_images.append(augmented_image)
            augmented_labels.append(augmented_label)
        
    augmented_images = np.array(augmented_images)
    augmented_labels = np.array(augmented_labels)

    return augmented_images, augmented_labels

#### Augmentation Deneyi
- Belirli sayıda augmented görüntü üreterek modelin veri artırma sonrası test başarısı incelenir. 

In [51]:
def run_augmented_experiment(new_images_per_class):

    augmented_images, augmented_labels = generate_augmented_dataset(train_images, train_labels, new_images_per_class)

    combined_train_images = np.concatenate((train_images , augmented_images), axis=0)

    combined_train_labels = np.concatenate((train_labels, augmented_labels), axis=0)

    combined_train_vectors = combined_train_images.reshape(combined_train_images.shape[0], 28 * 28)

    normalized_combined_train_vectors = combined_train_vectors / 255.0

    augmented_mean_vector, augmented_sorted_eigenvalues, augmented_sorted_eigenvectors = fit_manual_pca(normalized_combined_train_vectors)

    augmented_train_pca_vectors = transform_manual_pca(normalized_combined_train_vectors, augmented_mean_vector, augmented_sorted_eigenvectors, best_component_count)

    augmented_test_pca_vectors = transform_manual_pca(normalized_test_vectors, augmented_mean_vector, augmented_sorted_eigenvectors, best_component_count)

    augmented_accuracy = evaluate_knn(augmented_train_pca_vectors, combined_train_labels, augmented_test_pca_vectors, test_labels, best_k)

    augmented_preserved_variance = augmented_sorted_eigenvalues[:best_component_count].sum() / augmented_sorted_eigenvalues.sum()

    result = {
        "augmented_images_per_class": int(new_images_per_class),
        "total_augmented_images": int(augmented_images.shape[0]),
        "total_training_images": int(combined_train_images.shape[0]), 
        "component_count": int(best_component_count),
        "k": int(best_k),
        "preserved_variance" : float(augmented_preserved_variance * 100),
        "test_accuracy": float(augmented_accuracy * 100)
    }

    return result

#### Farklı Augmentation Miktarları

In [52]:
augmentation_options = [100, 500, 1000]
augmentation_results = []

for new_images_per_class in augmentation_options:
    result = run_augmented_experiment(new_images_per_class)
    augmentation_results.append(result)
    print(result)

{'augmented_images_per_class': 100, 'total_augmented_images': 1000, 'total_training_images': 61000, 'component_count': 50, 'k': 3, 'preserved_variance': 82.33024716194728, 'test_accuracy': 97.59}
{'augmented_images_per_class': 500, 'total_augmented_images': 5000, 'total_training_images': 65000, 'component_count': 50, 'k': 3, 'preserved_variance': 81.89052859715744, 'test_accuracy': 97.56}
{'augmented_images_per_class': 1000, 'total_augmented_images': 10000, 'total_training_images': 70000, 'component_count': 50, 'k': 3, 'preserved_variance': 81.47091225883769, 'test_accuracy': 97.53}


#### Baseline ve Augmentation Karşılaştırması

In [53]:
print("Baseline accuracy: ", round(baseline_result["test_accuracy"], 2))

for result in augmentation_results:
    difference = result["test_accuracy"] - baseline_result["test_accuracy"]

    print(
        result["augmented_images_per_class"], "per class",
        "| Total augmented:", result["total_augmented_images"],
        "| Accuracy:", round(result["test_accuracy"], 2),
        "| Difference:", round(difference, 2)
    )

Baseline accuracy:  97.52
100 per class | Total augmented: 1000 | Accuracy: 97.59 | Difference: 0.07
500 per class | Total augmented: 5000 | Accuracy: 97.56 | Difference: 0.04
1000 per class | Total augmented: 10000 | Accuracy: 97.53 | Difference: 0.01


#### En İyi Augmentation Sonucu

In [54]:
best_augmented_result = max(augmentation_results, key=lambda result: result["test_accuracy"])

best_difference = best_augmented_result["test_accuracy"] - baseline_result["test_accuracy"]

print("Best augmented result: ", best_augmented_result)
print("Best improvement over baseline: ", round(best_difference, 2))

Best augmented result:  {'augmented_images_per_class': 100, 'total_augmented_images': 1000, 'total_training_images': 61000, 'component_count': 50, 'k': 3, 'preserved_variance': 82.33024716194728, 'test_accuracy': 97.59}
Best improvement over baseline:  0.07
